# NHA Hackathon – Problem Statement 3  
## Skeletal Notebook: Document Forgery / Deepfake Detection

## Problem Understanding

According to the uploaded guidelines, PS3 requires participants to detect visible document tampering, classify each image/page using the predefined category IDs, and highlight manipulated regions using YAML annotations. The submission must contain:
1. a per-document or per-page **YAML annotation file**, and
2. a **JSON output** with `link`, `file_name`, and `Category_ID`.  

PDFs must be converted into per-page JPG-style entries such as `xyz_page_1.jpg` and produce corresponding YAML files such as `xyz_page_1.yaml`. Categories C8 and C10 are category-only cases with no YAML requirement, while other categories require bounding boxes in the exact YAML syntax expected by the organizers. fileciteturn1file0L1-L19 fileciteturn1file0L20-L30

## Categories in Scope

Use only the predefined category IDs:
- `C1` Copy-paste within the same document
- `C2` Overwriting on existing text
- `C3` Adding new content
- `C4` Removing / erasing text or image
- `C5` Merging content from different documents
- `C6` Watermark removal
- `C7` Irregular spacing
- `C8` Fully AI-generated document
- `C9` Partial AI-generated edits
- `C10` No editing / discrepancy found fileciteturn1file0L20-L30

In [40]:
!pip install -q opencv-python-headless pytesseract numpy scikit-learn scipy onnxruntime Pillow PyMuPDF PyYAML pandas

!pip install "optimum[onnxruntime]"

# If on the cloud Linux instance provided, we must install Hindi language packs for C2/C7 we will uncomment it while seding to the cloud Noteboollm
# !sudo apt-get install tesseract-ocr-hin -y

import os
import io
import json
import math
import asyncio
from dataclasses import dataclass, field, asdict
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import cv2
import numpy as np
import pytesseract
import scipy.fftpack
import yaml

try:
    from PIL import Image, ImageChops, ImageEnhance
except ImportError:
    Image = None

try:
    import fitz  # PyMuPDF for PDF conversion
except ImportError:
    fitz = None

try:
    import onnxruntime as ort # Staged for Tier 4 (Optional AI)
except ImportError:
    ort = None

# IMPORTANT: If testing on a local Windows machine, uncomment and set your Tesseract path.
# pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'

# Define standard paths and Category IDs per hackathon rules
INPUT_DIR = Path("./Claim_Documents")
OUTPUT_DIR = Path("./outputs")
ANNOTATIONS_DIR = OUTPUT_DIR / "annotations"
DEBUG_DIR = OUTPUT_DIR / "rendered_pages"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ANNOTATIONS_DIR.mkdir(parents=True, exist_ok=True)
DEBUG_DIR.mkdir(parents=True, exist_ok=True)

SUPPORTED_EXTS = {".jpg", ".jpeg", ".png", ".pdf"}

print("Environment Setup and Imports Successful.")


Environment Setup and Imports Successful.


## Download the Dataset
We have provided a dedicated widget to download the hackathon datasets directly from the platform into this notebook environment.

### 1. Import the Widget

from databank_download_widget import DatabankDownloadWidget

### 2. Download the Databank
Select the cell below and run it.
Enter the Databank ID for the hackathon package.
Enter your email and password for the platform.
Click the **Download** button.

The widget will download and unzip the data right into your current directory. You can monitor the progress in the status output area below the button.

### **Databank ID for PS3: 1ae9a4db-6b53-4a17-8274-fd3818c3f2be**

In [41]:
# from databank_download_widget import DatabankDownloadWidget

# # Create an instance of the databank widget
# databank_downloader = DatabankDownloadWidget()

# # Display the widget
# databank_downloader.display()

In [ ]:
# =========================
# 2. PATHS & CONFIG
# =========================
import shutil
from pathlib import Path

# Define standard paths
INPUT_DIR = Path("./Claim_Documents")
OUTPUT_DIR = Path("./outputs")
ANNOTATIONS_DIR = OUTPUT_DIR / "annotations"
DEBUG_DIR = OUTPUT_DIR / "rendered_pages"

# ---> THE FIX: Obliterate the old output folder if it exists <---
if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)

# Create fresh, empty directories
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ANNOTATIONS_DIR.mkdir(parents=True, exist_ok=True)
DEBUG_DIR.mkdir(parents=True, exist_ok=True)

SUPPORTED_EXTS = {".jpg", ".jpeg", ".png", ".pdf"}

print("Environment Setup, Imports, and Output Cleanup Successful.")

CATEGORY_IDS = {
    "C1": "Copy-paste within the same document",
    "C2": "Overwriting on existing text",
    "C3": "Adding new content",
    "C4": "Removing / erasing text or image",
    "C5": "Merging content from different documents",
    "C6": "Watermark removal",
    "C7": "Irregular spacing",
    "C8": "Fully AI-generated document",
    "C9": "Partial AI-generated edits",
    "C10": "No editing / discrepancy found",
}

CATEGORY_ONLY_CLASSES = {"C8", "C10"}

Environment Setup and Imports Successful.


In [43]:
from pathlib import Path

# Define standard paths
INPUT_DIR = Path("./Claim_Documents")
OUTPUT_DIR = Path("./outputs")
ANNOTATIONS_DIR = OUTPUT_DIR / "annotations"
DEBUG_DIR = OUTPUT_DIR / "rendered_pages"

# Create directories
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ANNOTATIONS_DIR.mkdir(parents=True, exist_ok=True)
DEBUG_DIR.mkdir(parents=True, exist_ok=True)

# Fixed variable name to match the onboarding helper logic
SUPPORTED_EXTS = {".jpg", ".jpeg", ".png", ".pdf"}

In [44]:
!optimum-cli export onnx --model dima806/deepfake_vs_real_image_detection --task image-classification hf_model_export/

`torch_dtype` is deprecated! Use `dtype` instead!
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
/home/shivaji/Desktop/PMJAY_AXIOM/.venv/lib/python3.13/site-packages/transformers/models/vit/feature_extraction_vit.py:30: FutureWarning: The class ViTFeatureExtractor is deprecated and will be removed in version 5 of Transformers. Please use ViTImageProcessor instead.
  warnings.warn(
/home/shivaji/Desktop/PMJAY_AXIOM/.venv/lib/python3.13/site-packages/transformers/models/vit/modeling_vit.py:155: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trac

In [45]:
# =========================
# 4. DATA STRUCTURES
# =========================

@dataclass
class DocumentPage:
    source_link: str
    original_path: str
    original_file_name: str
    page_file_name: str
    page_number: int
    is_pdf: bool
    image_path: Optional[str] = None
    image_width: Optional[int] = None
    image_height: Optional[int] = None


@dataclass
class DetectedRegion:
    x: int
    y: int
    w: int
    h: int
    category_id: str
    type: Optional[str] = None
    stretch_factor: Optional[float] = None
    header_source: Optional[str] = None
    body_source: Optional[str] = None


@dataclass
class PageAnalysisResult:
    source_link: str
    file_name: str
    original_file_name: str
    page_number: int
    predicted_categories: List[str] = field(default_factory=list)
    detected_regions: List[DetectedRegion] = field(default_factory=list)
    notes: Dict[str, Any] = field(default_factory=dict)
    debug: Dict[str, Any] = field(default_factory=dict)

In [46]:

SUPPORTED_IMAGE_EXTS = {".jpg", ".jpeg", ".png"}
SUPPORTED_PDF_EXTS = {".pdf"}

def safe_open_image_size(path):
    try:
        if Image:
            with Image.open(path) as img:
                return img.size
        else:
            img = cv2.imread(str(path))
            if img is not None:
                return img.shape[1], img.shape[0]
    except:
        pass
    return None, None

def render_pdf_to_images(pdf_path, render_dir):
    pages = []
    if not fitz:
        return pages
    try:
        doc = fitz.open(pdf_path)
        for i in range(len(doc)):
            page = doc.load_page(i)
            pix = page.get_pixmap()
            img_path = render_dir / f"{pdf_path.stem}_page_{i+1}.png"
            pix.save(str(img_path))
            pages.append(
                DocumentPage(
                    source_link=str(pdf_path),
                    original_path=str(pdf_path),
                    original_file_name=pdf_path.name,
                    page_file_name=img_path.name,
                    page_number=i+1,
                    is_pdf=True,
                    image_path=str(img_path),
                    image_width=pix.width,
                    image_height=pix.height,
                )
            )
    except:
        pass
    return pages

# =========================
# PIPELINE STAGES (F1-OPTIMIZED WITH NHA STUBS)
# =========================
import os
import cv2
import json
import base64
import numpy as np
import pytesseract
from pathlib import Path
from PIL import Image, ImageChops, ImageEnhance
from typing import List, Dict, Any, Optional

try:
    import onnxruntime as ort
except ImportError:
    ort = None
def build_document_pages(input_dir: Path, render_dir: Path) -> List[DocumentPage]:
    pages: List[DocumentPage] = []
    render_dir.mkdir(parents=True, exist_ok=True)

    for file_path in sorted(input_dir.iterdir()):
        if not file_path.is_file():
            continue

        suffix = file_path.suffix.lower()

        if suffix in SUPPORTED_IMAGE_EXTS:
            width, height = safe_open_image_size(file_path)
            pages.append(
                DocumentPage(
                    source_link=str(file_path),
                    original_path=str(file_path),
                    original_file_name=file_path.name,
                    page_file_name=file_path.name,
                    page_number=1,
                    is_pdf=False,
                    image_path=str(file_path),
                    image_width=width,
                    image_height=height,
                )
            )
        elif suffix in SUPPORTED_PDF_EXTS:
            pages.extend(render_pdf_to_images(file_path, render_dir))

    return pages
# ---------------------------------------------------------
# HELPER MATH ENGINES (FOR PRECISE BOUNDING BOXES)
# ---------------------------------------------------------
def calculate_iou(boxA: List[int], boxB: List[int]) -> float:
    xA, yA = max(boxA[0], boxB[0]), max(boxA[1], boxB[1])
    xB, yB = min(boxA[0] + boxA[2], boxB[0] + boxB[2]), min(boxA[1] + boxA[3], boxB[1] + boxB[3])
    interArea = max(0, xB - xA) * max(0, yB - yA)
    if interArea == 0: return 0.0
    return interArea / float((boxA[2] * boxA[3]) + (boxB[2] * boxB[3]) - interArea)

def run_tier1_ocr_math(img_path: str) -> List[DetectedRegion]:
    regions = []
    img = cv2.imread(img_path)
    if img is None: return regions
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    try:
        d = pytesseract.image_to_data(gray, lang='eng+hin', config=r'--oem 3 --psm 11', output_type=pytesseract.Output.DICT)
    except Exception:
        try:
            # FALLBACK: If Hindi is missing on the machine, fall back to English to prevent crashing
            d = pytesseract.image_to_data(gray, lang='eng', config=r'--oem 3 --psm 11', output_type=pytesseract.Output.DICT)
        except: return regions
            
    boxes = [[d['left'][i], d['top'][i], d['width'][i], d['height'][i]] for i in range(len(d['level'])) if int(d['conf'][i]) > 15 and d['text'][i].strip()]
    
    # C2 Check
    for i in range(len(boxes)):
        for j in range(i + 1, len(boxes)):
            if calculate_iou(boxes[i], boxes[j]) > 0.01:
                x, y = min(boxes[i][0], boxes[j][0]), min(boxes[i][1], boxes[j][1])
                w, h = max(boxes[i][0]+boxes[i][2], boxes[j][0]+boxes[j][2]) - x, max(boxes[i][1]+boxes[i][3], boxes[j][1]+boxes[j][3]) - y
                regions.append(DetectedRegion(x=int(x), y=int(y), w=int(w), h=int(h), category_id="C2"))

    # C7 Check
    boxes.sort(key=lambda b: b[1])
    spacing_data = [(boxes[i+1][0] - (boxes[i][0] + boxes[i][2]), boxes[i+1]) for i in range(len(boxes)-1) if abs(boxes[i][1] - boxes[i+1][1]) < 15 and (boxes[i+1][0] - (boxes[i][0] + boxes[i][2])) > 0]
    if spacing_data:
        spaces = [s[0] for s in spacing_data]
        med, std = np.median(spaces), np.std(spaces)
        for space, box in spacing_data:
            if space > 80 or (med > 0 and space > med + 1.5 * std):
                regions.append(DetectedRegion(x=int(box[0]-space), y=int(box[1]), w=int(space+box[2]), h=int(box[3]), category_id="C7"))
    return regions

def run_tier2_sift(img_path: str) -> List[DetectedRegion]:
    regions = []
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    if img is None: return regions
    kp, des = cv2.SIFT_create().detectAndCompute(img, None)
    if des is not None and len(kp) > 50:
        matches = cv2.FlannBasedMatcher(dict(algorithm=1, trees=5), dict(checks=50)).knnMatch(des, des, k=3)
        good = [m[0] if m[0].queryIdx != m[0].trainIdx else m[1] for m in matches if len(m) >= 2]
        good = [m for m in good if np.linalg.norm(np.array(kp[m.queryIdx].pt) - np.array(kp[m.trainIdx].pt)) > 100 and m.distance < 0.75]
        if len(good) > 15:
            x, y, w, h = cv2.boundingRect(np.float32([kp[m.queryIdx].pt for m in good]).reshape(-1, 2))
            regions.append(DetectedRegion(x=x, y=y, w=w, h=h, category_id="C1"))
    return regions

def run_tier3_ela(img_path: str) -> List[DetectedRegion]:
    regions = []
    try:
        img_bgr = cv2.imread(img_path)
        original = Image.open(img_path).convert('RGB')
        original.save("temp.jpg", 'JPEG', quality=75)
        diff = ImageChops.difference(original, Image.open("temp.jpg"))
        os.remove("temp.jpg")
        
        ela_map = ImageEnhance.Brightness(diff).enhance(255.0 / (max([ex[1] for ex in diff.getextrema()]) or 1))
        _, thresh = cv2.threshold(cv2.cvtColor(np.array(ela_map), cv2.COLOR_RGB2GRAY), 150, 255, cv2.THRESH_BINARY)
        contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        
        gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
        h, w = gray.shape
        for c in contours:
            x, y, bw, bh = cv2.boundingRect(c)
            if bw * bh > 100:
                avg = np.mean(gray[y:y+bh, x:x+bw])
                if bw * bh > (h * w * 0.10) and avg > 190: cat = "C6"
                else: cat = "C4" if avg > 200 else "C3"
                regions.append(DetectedRegion(x=x, y=y, w=bw, h=bh, category_id=cat))
                
        # C5 (Stitched Documents)
        if abs(np.median(gray[0:h//2, :]) - np.median(gray[h//2:h, :])) > 8:
            regions.append(DetectedRegion(x=0, y=h//2, w=w, h=h//2, category_id="C5"))
    except: pass
    return regions

# ---------------------------------------------------------
# REQUIRED HACKATHON STUBS
# ---------------------------------------------------------

async def run_nha_vision_prompt(image_path: str, prompt_text: str, model_name: str = "bedrock/converse/amazon.nova-lite-v1:0") -> Dict[str, Any]:
    """
    Simulated API call. In a real environment with credentials, this would use aiohttp to hit Amazon Bedrock.
    Because VLMs fail at exact bounding box math, we rely on the OpenCV engines for the actual logic.
    """
    try:
        with open(image_path, "rb") as image_file:
            encoded_string = base64.b64encode(image_file.read()).decode('utf-8')
    except Exception:
        pass
    # Returning a safe dummy structure. OpenCV will overwrite this.
    return {"status": "simulated_success", "predicted_categories": ["C10"]}


def assess_page_quality(page: DocumentPage) -> Dict[str, Any]:
    if not page.image_path: return {"status": "error"}
    img = cv2.imread(page.image_path, cv2.IMREAD_GRAYSCALE)
    if img is None: return {"status": "error"}
    
    lap_var = cv2.Laplacian(img, cv2.CV_64F).var()
    return {
        "laplacian_variance": round(lap_var, 2),
        "is_blurry": bool(lap_var < 100.0),
        "status": "ok"
    }

async def detect_tampering_categories(page: DocumentPage) -> List[str]:
    categories = []
    if not page.image_path: return ["C10"]
    
    # Global AI Checks (C8/C9) using the Pre-trained ViT
    try:
        # 1. FIX: Read as BGR color, not Grayscale, because the ViT expects RGB input
        img_bgr = cv2.imread(page.image_path, cv2.IMREAD_COLOR)
        if img_bgr is None: return ["C10"]
        
        model_path = "./hf_model_export/model.onnx"
        if ort is not None and Path(model_path).exists():
            session = ort.InferenceSession(model_path, providers=['CPUExecutionProvider'])
            input_name = session.get_inputs()[0].name
            
            def check_ai_crop(crop_img_bgr):
                # 2. FIX: Convert BGR to RGB
                rgb_crop = cv2.cvtColor(crop_img_bgr, cv2.COLOR_BGR2RGB)
                
                # 3. FIX: Resize and apply Standard ImageNet Normalization
                resized = cv2.resize(rgb_crop, (224, 224)).astype(np.float32) / 255.0
                mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
                std = np.array([0.229, 0.224, 0.225], dtype=np.float32)
                normalized = (resized - mean) / std
                
                # Transpose to (Channels, Height, Width) and add Batch dimension
                tensor = np.transpose(normalized, (2, 0, 1))
                tensor = np.expand_dims(tensor, axis=0)
                
                # Infer
                logits = session.run(None, {input_name: tensor})[0][0]
                
                # The dima806 model outputs [Real_Logit, Fake_Logit]
                # If Fake (Index 1) > Real (Index 0), flag it.
                if len(logits) > 1:
                    return float(logits[1]) > float(logits[0])
                else:
                    return float(logits) > 0.80

            # Check full image for C8 (Fully AI)
            if check_ai_crop(img_bgr): 
                categories.append("C8")
                
            # Check Quadrants for C9 (Partial AI)
            h, w, _ = img_bgr.shape
            quadrants = [
                img_bgr[0:h//2, 0:w//2], img_bgr[0:h//2, w//2:w], 
                img_bgr[h//2:h, 0:w//2], img_bgr[h//2:h, w//2:w]
            ]
            ai_quadrants_count = sum([1 for q in quadrants if check_ai_crop(q)])
            
            if 0 < ai_quadrants_count < 4:
                categories.append("C9")
                
    except Exception as e:
        print(f"Tier 4 AI Analysis Error on {page.page_file_name}: {e}")
    
    return categories if categories else ["C10"]

async def localize_tampered_regions(page: DocumentPage, predicted_categories: List[str]) -> List[DetectedRegion]:
    regions = []
    if not page.image_path: return regions
    
    # We execute our deterministic CV engines here to guarantee precise YAML bounding boxes.
    regions.extend(run_tier1_ocr_math(page.image_path))
    regions.extend(run_tier2_sift(page.image_path))
    regions.extend(run_tier3_ela(page.image_path))
    
    return regions

def postprocess_regions(regions: List[DetectedRegion], predicted_categories: List[str], page: DocumentPage) -> List[DetectedRegion]:
    if not regions: return []
    img_w = page.image_width or float('inf')
    img_h = page.image_height or float('inf')
    
    valid = []
    for r in regions:
        x1, y1 = max(0, r.x), max(0, r.y)
        w, h = min(img_w, r.x + r.w) - x1, min(img_h, r.y + r.h) - y1
        if w > 0 and h > 0:
            r.x, r.y, r.w, r.h = int(x1), int(y1), int(w), int(h)
            valid.append(r)

    merged = []
    grouped = {}
    for r in valid: grouped.setdefault(r.category_id, []).append(r)

    for cat_id, boxes in grouped.items():
        active = boxes.copy()
        while active:
            curr = active.pop(0)
            was_merged = False
            for i, other in enumerate(active):
                if calculate_iou([curr.x, curr.y, curr.w, curr.h], [other.x, other.y, other.w, other.h]) > 0.10:
                    nx, ny = min(curr.x, other.x), min(curr.y, other.y)
                    curr.w, curr.h = max(curr.x + curr.w, other.x + other.w) - nx, max(curr.y + curr.h, other.y + other.h) - ny
                    curr.x, curr.y = nx, ny
                    active.pop(i)
                    active.append(curr)
                    was_merged = True
                    break
            if not was_merged: merged.append(curr)
    
    # Remove strict duplicates
    final, seen = [], set()
    for r in merged:
        sig = (r.x, r.y, r.w, r.h, r.category_id)
        if sig not in seen:
            seen.add(sig)
            final.append(r)
    return final

def enrich_region_metadata(regions: List[DetectedRegion], predicted_categories: List[str], page: DocumentPage) -> List[DetectedRegion]:
    for r in regions:
        if r.category_id == "C3": r.type = "text"
        elif r.category_id == "C4": r.type = "erased"
        elif r.category_id == "C7": 
            r.type = "irregular_spacing"
            r.stretch_factor = 1.25
        elif r.category_id == "C5":
            r.header_source, r.body_source = "source_A", "stitched_B"
        elif r.category_id == "C9": r.type = "ai_generated_field"
    return regions

def validate_prediction(page: DocumentPage, predicted_categories: List[str], regions: List[DetectedRegion]) -> Dict[str, Any]:
    VALID_CATS = {"C1", "C2", "C3", "C4", "C5", "C6", "C7", "C8", "C9", "C10"}
    CAT_ONLY = {"C8", "C10"}
    
    region_cats = set(r.category_id for r in regions)
    all_cats = set(predicted_categories).union(region_cats).intersection(VALID_CATS)
    
    if len(all_cats) > 1 and "C10" in all_cats: all_cats.remove("C10")
    if not all_cats: all_cats.add("C10")
    
    # Sort: Place global C8 first, then sort remaining by area size
    cat_areas = {cat: sum(r.w * r.h for r in regions if r.category_id == cat) for cat in all_cats}
    localized = sorted([c for c in all_cats if c not in CAT_ONLY], key=lambda c: cat_areas.get(c, 0), reverse=True)
    
    final_ordered = []
    if "C8" in all_cats: final_ordered.append("C8")
    final_ordered.extend(localized)
    if "C10" in all_cats and "C10" not in final_ordered: final_ordered.append("C10")

    return {
        "valid": True,
        "final_categories": final_ordered,
        "dropped_categories": [],
        "notes": "Validation complete."
    }

def build_page_analysis_result(page: DocumentPage, predicted_categories: List[str], regions: List[DetectedRegion], quality_summary: Optional[Dict[str, Any]] = None, validation: Optional[Dict[str, Any]] = None) -> PageAnalysisResult:
    final_cats = validation.get("final_categories", ["C10"]) if validation else predicted_categories
    return PageAnalysisResult(
        source_link=page.source_link,
        file_name=page.page_file_name,
        original_file_name=page.original_file_name,
        page_number=page.page_number,
        predicted_categories=final_cats,
        detected_regions=regions,
        notes={"pipeline": "cv_hybrid_engine"},
        debug={"quality": quality_summary, "validation": validation}
    )

In [47]:
# import pytesseract

# def calculate_iou(boxA: List[int], boxB: List[int]) -> float:
#     xA = max(boxA[0], boxB[0])
#     yA = max(boxA[1], boxB[1])
#     xB = min(boxA[0] + boxA[2], boxB[0] + boxB[2])
#     yB = min(boxA[1] + boxA[3], boxB[1] + boxB[3])
#     interArea = max(0, xB - xA) * max(0, yB - yA)
#     if interArea == 0: return 0.0
#     return interArea / float((boxA[2] * boxA[3]) + (boxB[2] * boxB[3]) - interArea)

# def run_tier1_spatial_math(page: DocumentPage) -> List[DetectedRegion]:
#     regions = []
#     if not page.image_path: return regions
#     img = cv2.imread(page.image_path)
#     if img is None: return regions
#     gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
#     custom_config = r'--oem 3 --psm 11'
#     try:
#         d = pytesseract.image_to_data(gray, config=custom_config, output_type=pytesseract.Output.DICT)
#     except Exception:
#         return regions
    
#     boxes = []
#     for i in range(len(d['level'])):
#         if int(d['conf'][i]) > 40 and d['text'][i].strip() != '':
#             boxes.append([d['left'][i], d['top'][i], d['width'][i], d['height'][i]])
            
#     # C2 (Overwriting)
#     for i in range(len(boxes)):
#         for j in range(i + 1, len(boxes)):
#             if calculate_iou(boxes[i], boxes[j]) > 0.05:
#                 x_min, y_min = min(boxes[i][0], boxes[j][0]), min(boxes[i][1], boxes[j][1])
#                 x_max, y_max = max(boxes[i][0]+boxes[i][2], boxes[j][0]+boxes[j][2]), max(boxes[i][1]+boxes[i][3], boxes[j][1]+boxes[j][3])
#                 regions.append(DetectedRegion(x=int(x_min), y=int(y_min), w=int(x_max - x_min), h=int(y_max - y_min), category_id="C2"))

#     # C7 (Irregular Spacing)
#     boxes.sort(key=lambda b: b[1])
#     spacing_data = []
#     for i in range(len(boxes) - 1):
#         if abs(boxes[i][1] - boxes[i+1][1]) < 15:
#             space = boxes[i+1][0] - (boxes[i][0] + boxes[i][2])
#             if space > 0: spacing_data.append((space, boxes[i+1]))
                
#     if spacing_data:
#         spaces = [s[0] for s in spacing_data]
#         median_space, std_space = np.median(spaces), np.std(spaces)
#         if median_space > 0 and std_space > 0:
#             for space, box in spacing_data:
#                 if space > (median_space + (3 * std_space)):
#                     stretch = round(float(space / median_space), 2)
#                     if stretch < 15.0: 
#                         regions.append(DetectedRegion(x=int(box[0] - space), y=int(box[1]), w=int(space + box[2]), h=int(box[3]), category_id="C7", type="irregular_spacing", stretch_factor=stretch))
#     return regions

In [48]:
# from PIL import Image, ImageChops, ImageEnhance
# import os

# def run_tier3_ela_contours(page: DocumentPage) -> List[DetectedRegion]:
#     """
#     Tier 3 Scanner: Targets C3 (Added), C4 (Erased), and C6 (Watermark Removed).
#     """
#     regions = []
#     if not page.image_path:
#         return regions
        
#     try:
#         # 1. GENERATE THE ELA MAP
#         original = Image.open(page.image_path).convert('RGB')
#         temp_filename = f"temp_ela_{page.page_file_name}.jpg"
#         original.save(temp_filename, 'JPEG', quality=90)
#         compressed = Image.open(temp_filename)
        
#         ela_diff = ImageChops.difference(original, compressed)
#         extrema = ela_diff.getextrema()
#         max_diff = max([ex[1] for ex in extrema])
#         if max_diff == 0:
#             max_diff = 1
#         scale = 255.0 / max_diff
#         ela_map = ImageEnhance.Brightness(ela_diff).enhance(scale)
#         os.remove(temp_filename)
        
#         # 2. OPENCV CONTOUR DETECTION
#         ela_cv = np.array(ela_map)
#         ela_gray = cv2.cvtColor(ela_cv, cv2.COLOR_RGB2GRAY)
#         _, thresh = cv2.threshold(ela_gray, 240, 255, cv2.THRESH_BINARY)
#         kernel = np.ones((5,5), np.uint8)
#         thresh = cv2.dilate(thresh, kernel, iterations=2)
#         contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        
#         original_cv = cv2.imread(page.image_path)
#         original_gray = cv2.cvtColor(original_cv, cv2.COLOR_BGR2GRAY)
#         height, width = original_gray.shape
#         page_area = height * width
        
#         for contour in contours:
#             x, y, w, h = cv2.boundingRect(contour)
#             area = w * h
#             if area < 400: 
#                 continue
                
#             roi_gray = original_gray[y:y+h, x:x+w]
#             avg_intensity = np.mean(roi_gray)
            
#             if area > (page_area * 0.10) and avg_intensity > 230:
#                 regions.append(DetectedRegion(x=x, y=y, w=w, h=h, category_id="C6"))
#                 continue
#             if avg_intensity > 240:
#                 regions.append(DetectedRegion(x=x, y=y, w=w, h=h, category_id="C4", type="erased"))
#             elif avg_intensity <= 240:
#                 regions.append(DetectedRegion(x=x, y=y, w=w, h=h, category_id="C3", type="text"))

#     except Exception as e:
#         print(f"Tier 3 ELA Error on {page.page_file_name}: {e}")
        
#     return regions

In [49]:
# # =========================
# # PIPELINE STAGES
# # =========================


# # async def run_nha_vision_prompt(
# #     image_path: str,
# #     prompt_text: str,
# #     model_name: str = "bedrock/converse/amazon.nova-lite-v1:0",
# # ) -> Dict[str, Any]:
# #     """
# #     Call the NHA client on a page image using a task-specific prompt.

# #     Intended responsibilities:
# #     - Open the image file from disk and convert it to a base64 data URL.
# #     - Send the image and prompt to the mandatory NHA Bedrock model.
# #     - Return the raw or lightly normalized response for downstream parsing.
# #     - Serve as the common model-calling helper reused across all PS3 subtasks.
# #     - Handle API failures, response parsing errors, and retry logic if needed.
# #     """
# #     pass


# def assess_page_quality(page: DocumentPage) -> Dict[str, Any]:
#     """
#     Assess whether the page image is suitable for tampering analysis using 
#     CPU-bound classical computer vision (Laplacian Variance).

#     Intended responsibilities:
#     - Estimate blur and heavy compression without using ML models.
#     - Flag low-quality pages that may reduce confidence.
#     - Return a structured quality summary.
#     """
#     if not page.image_path:
#         return {"status": "error", "reason": "No image path provided."}
    
#     # Read the image in grayscale (faster and requires less memory)
#     img = cv2.imread(page.image_path, cv2.IMREAD_GRAYSCALE)
    
#     if img is None:
#         return {"status": "error", "reason": "Could not read image file."}
        
#     # Calculate the variance of the Laplacian to measure edge sharpness
#     laplacian_var = cv2.Laplacian(img, cv2.CV_64F).var()
    
#     # Standard threshold: < 100 usually indicates a blurry image
#     is_blurry = bool(laplacian_var < 100.0)
    
#     # Optional: Check for extreme skew/brightness issues here if needed in the future,
#     # but Laplacian variance gives us the best bang-for-buck on a 4-core CPU.
    
#     return {
#         "laplacian_variance": round(laplacian_var, 2),
#         "is_blurry": is_blurry,
#         "status": "ok",
#         "notes": "Image is blurry" if is_blurry else "Image quality acceptable"
#     }


# async def detect_tampering_categories(page: DocumentPage) -> List[str]:
#     """
#     Tier 4 Global Check: Predicts page-level forgery categories (C8).
    
#     Intended responsibilities:
#     - Runs a Fast Fourier Transform (FFT) to extract the frequency spectrum.
#     - Passes the spectrum to an INT8 Quantized ONNX classifier.
#     - Returns ["C8"] if fully AI-generated, otherwise returns empty list [].
#     """
#     categories = []
#     if not page.image_path or ort is None:
#         return categories
        
#     try:
#         # 1. Read image in grayscale to save memory
#         img = cv2.imread(page.image_path, cv2.IMREAD_GRAYSCALE)
#         if img is None:
#             return categories
            
#         # 2. Fast Fourier Transform (FFT) to expose AI frequency artifacts
#         f_transform = np.fft.fft2(img)
#         f_shift = np.fft.fftshift(f_transform)
        
#         # Calculate magnitude spectrum 
#         magnitude_spectrum = 20 * np.log(np.abs(f_shift) + 1e-8)
        
#         # ---> THE FIX: Pure Numpy Normalization (Bypasses OpenCV completely)
#         min_val = np.min(magnitude_spectrum)
#         max_val = np.max(magnitude_spectrum)
        
#         if max_val > min_val:
#             normalized_img = (magnitude_spectrum - min_val) / (max_val - min_val) * 255.0
#         else:
#             normalized_img = np.zeros_like(magnitude_spectrum)
            
#         spectrum_img = np.uint8(normalized_img)
        
#         # ---> THE VIT FIX: Convert to 3-channel (RGB) before resizing
#         spectrum_img_3c = cv2.cvtColor(spectrum_img, cv2.COLOR_GRAY2BGR)
        
#         # Resize to match standard lightweight model input (224x224)
#         input_tensor = cv2.resize(spectrum_img_3c, (224, 224))
        
#         # Prepare for ONNX (PyTorch models expect Channel-First format: Channels, Height, Width)
#         input_tensor = input_tensor.astype(np.float32) / 255.0
#         input_tensor = np.transpose(input_tensor, (2, 0, 1)) # Changes shape to (3, 224, 224)
        
#         # Add batch dimension -> Final shape: [1, 3, 224, 224]
#         input_tensor = np.expand_dims(input_tensor, axis=0)
        
#         # 3. Quantized ONNX Inference (Extremely fast on CPU)
#         model_path = "./models/efficientnet_lite_int8.onnx"
        
#         if Path(model_path).exists():
#             session = ort.InferenceSession(model_path, providers=['CPUExecutionProvider'])
#             input_name = session.get_inputs()[0].name
            
#             # Run the model
#             outputs = session.run(None, {input_name: input_tensor})
            
#             # Hugging Face ViT Real/Fake models usually output 2 logits (Real, Fake)
#             logits = outputs[0][0]
#             if len(logits) > 1:
#                 # If Fake logit is higher than Real logit, flag it
#                 if float(logits[1]) > float(logits[0]):
#                     categories.append("C8")
#             else:
#                 if float(logits) > 0.85:
#                     categories.append("C8")
                
#     except Exception as e:
#         print(f"Tier 4 Frequency Analysis Error on {page.page_file_name}: {e}")

#     return categories


# async def localize_tampered_regions(
#     page: DocumentPage,
#     predicted_categories: List[str],
# ) -> List[DetectedRegion]:
#     """
#     Detect and localize suspicious regions for categories that require annotations.

#     Intended responsibilities:
#     - Produce bounding boxes for tampered areas on the page.
#     - Associate each bounding box with the corresponding category ID.
#     - Ensure boxes tightly cover the manipulated text, stamp, signature, image region,
#       spacing anomaly, watermark region, or edited field.
#     - Skip localization for category-only classes where YAML is not required.
#     """
#     all_regions = []
    
#     # 1. RUN TIER 1: Spatial Math (OCR bounding box math for C2, C7)
#     try:
#         tier1_results = run_tier1_spatial_math(page)
#         all_regions.extend(tier1_results)
#     except Exception as e:
#         print(f"Tier 1 Error on {page.page_file_name}: {e}")

#     # 2. RUN TIER 3: Visual Forensics (ELA Contours for C3, C4, C6)
#     try:
#         tier3_results = run_tier3_ela_contours(page)
#         all_regions.extend(tier3_results)
#     except Exception as e:
#         print(f"Tier 3 Error on {page.page_file_name}: {e}")
        
#     # Note: Tier 2 (SIFT) would be injected here if you decide to activate C1 logic.
    
#     return all_regions


# def postprocess_regions(
#     regions: List[DetectedRegion],
#     predicted_categories: List[str],
#     page: DocumentPage,
# ) -> List[DetectedRegion]:
#     """
#     Clean and normalize the raw localized regions before export.

#     Intended responsibilities:
#     - Remove invalid or duplicate boxes.
#     - Clip coordinates to the image boundaries.
#     - Merge overlapping boxes when appropriate.
#     - Preserve category-specific metadata such as type, stretch_factor,
#       header_source, or body_source when required by sample syntax.
#     """
#     if not regions:
#         return []

#     # 1. Clip coordinates to image boundaries and remove invalid boxes
#     img_w = page.image_width if page.image_width is not None else float('inf')
#     img_h = page.image_height if page.image_height is not None else float('inf')
    
#     valid_regions = []
#     for r in regions:
#         # Clip to boundaries
#         x1 = max(0, r.x)
#         y1 = max(0, r.y)
#         x2 = min(img_w, r.x + r.w)
#         y2 = min(img_h, r.y + r.h)
        
#         new_w = int(x2 - x1)
#         new_h = int(y2 - y1)
        
#         # Only keep if it has actual area
#         if new_w > 0 and new_h > 0:
#             r.x = int(x1)
#             r.y = int(y1)
#             r.w = new_w
#             r.h = new_h
#             valid_regions.append(r)

#     # Helper function for Intersection over Union (IoU)
#     def get_iou(box1: DetectedRegion, box2: DetectedRegion) -> float:
#         x_left = max(box1.x, box2.x)
#         y_top = max(box1.y, box2.y)
#         x_right = min(box1.x + box1.w, box2.x + box2.w)
#         y_bottom = min(box1.y + box1.h, box2.y + box2.h)
        
#         if x_right < x_left or y_bottom < y_top:
#             return 0.0
            
#         intersection = (x_right - x_left) * (y_bottom - y_top)
#         area1 = box1.w * box1.h
#         area2 = box2.w * box2.h
#         return intersection / float(area1 + area2 - intersection)

#     # 2. Merge overlapping boxes of the SAME category
#     merged_regions = []
    
#     # Group by category ID to avoid merging a C2 box with a C3 box
#     grouped_regions = {}
#     for r in valid_regions:
#         grouped_regions.setdefault(r.category_id, []).append(r)

#     for cat_id, cat_boxes in grouped_regions.items():
#         active_boxes = cat_boxes.copy()
        
#         while active_boxes:
#             current = active_boxes.pop(0)
#             merged = False
            
#             for i, other in enumerate(active_boxes):
#                 # If they overlap by more than 20%, merge them
#                 if get_iou(current, other) > 0.20:
#                     # Create union bounding box
#                     new_x = min(current.x, other.x)
#                     new_y = min(current.y, other.y)
#                     new_w = max(current.x + current.w, other.x + other.w) - new_x
#                     new_h = max(current.y + current.h, other.y + other.h) - new_y
                    
#                     current.x = new_x
#                     current.y = new_y
#                     current.w = new_w
#                     current.h = new_h
                    
#                     # 3. Preserve category-specific metadata
#                     current.type = current.type or other.type
#                     current.stretch_factor = current.stretch_factor or other.stretch_factor
#                     current.header_source = current.header_source or other.header_source
#                     current.body_source = current.body_source or other.body_source
                    
#                     # Remove the box we just consumed and put the updated union box back 
#                     # in the queue to check against remaining boxes
#                     active_boxes.pop(i)
#                     active_boxes.append(current)
#                     merged = True
#                     break
            
#             if not merged:
#                 merged_regions.append(current)

#     # 4. Final Deduplication (Catch-all for exact identical duplicates)
#     final_regions = []
#     seen = set()
#     for r in merged_regions:
#         sig = (r.x, r.y, r.w, r.h, r.category_id, r.type, r.stretch_factor)
#         if sig not in seen:
#             seen.add(sig)
#             final_regions.append(r)

#     return final_regions


# def enrich_region_metadata(
#     regions: List[DetectedRegion],
#     predicted_categories: List[str],
#     page: DocumentPage,
# ) -> List[DetectedRegion]:
#     """
#     Add category-specific annotation metadata required by the organizer examples.

#     Intended responsibilities:
#     - For C3, optionally tag regions with type values such as text, stamp, or signature.
#     - For C4, optionally tag erased regions with a type indicating removed content.
#     - For C5, optionally include body_source or header_source values when relevant.
#     - For C7, optionally include stretch_factor and a type such as irregular_spacing.
#     - For C9, optionally indicate edited field type such as name, date, or amount.
#     """
#     for r in regions:
#         # C3: Added Content Failsafe
#         if r.category_id == "C3" and r.type is None:
#             r.type = "text" 
            
#         # C4: Erased Content Failsafe
#         elif r.category_id == "C4" and r.type is None:
#             r.type = "erased"
            
#         # C7: Irregular Spacing Failsafe
#         elif r.category_id == "C7":
#             if r.type is None:
#                 r.type = "irregular_spacing"
#             # Our Tier 1 engine calculates the exact stretch_factor. 
#             # This is just a dummy fallback to prevent YAML syntax errors if Tier 1 hiccupped.
#             if r.stretch_factor is None:
#                 r.stretch_factor = 1.15 
                
#         # C5: Merged Documents Failsafe
#         elif r.category_id == "C5":
#             if r.body_source is None:
#                 r.body_source = "unknown_source_A"
#             if r.header_source is None:
#                 r.header_source = "unknown_source_B"
                
#         # C9: Partial AI Failsafe
#         elif r.category_id == "C9" and r.type is None:
#             r.type = "ai_generated_field"

#     return regions


# def validate_prediction(
#     page: DocumentPage,
#     predicted_categories: List[str],
#     regions: List[DetectedRegion],
# ) -> Dict[str, Any]:
#     """
#     Validate that the page prediction satisfies submission requirements.

#     Intended responsibilities:
#     - Ensure that only valid category IDs are used.
#     - Enforce that localization exists when the selected category requires YAML.
#     - Ensure that category-only classes do not force unnecessary annotations.
#     - Confirm that multi-label ordering puts the dominant manipulation first.
#     """
#     VALID_CATS = {"C1", "C2", "C3", "C4", "C5", "C6", "C7", "C8", "C9", "C10"}
#     CATEGORY_ONLY_CLASSES = {"C8", "C10"}
    
#     # 1. Gather all predicted categories (from Regions + Global checks like Tier 4 C8)
#     region_cats = set(r.category_id for r in regions)
#     global_cats = set(predicted_categories)
#     all_cats = region_cats.union(global_cats)
    
#     # RULE: Ensure that only valid category IDs are used
#     all_cats = {c for c in all_cats if c in VALID_CATS}
    
#     # RULE: Enforce that localization exists when the selected category requires YAML
#     validated_cats = set()
#     for cat in all_cats:
#         if cat in CATEGORY_ONLY_CLASSES:
#             validated_cats.add(cat) # C8 and C10 don't need regions
#         elif cat in region_cats:
#             validated_cats.add(cat) # C1-C7, C9 must have at least one valid region
            
#     # RULE: Ensure category-only classes do not force annotations (C10 Exclusivity)
#     # If we found actual tampering (like C2 or C3), the document is NOT clean (C10).
#     if len(validated_cats) > 1 and "C10" in validated_cats:
#         validated_cats.remove("C10")
#     # Failsafe: If no tampering is found at all, it MUST be C10.
#     elif len(validated_cats) == 0:
#         validated_cats.add("C10")
        
#     # RULE: Confirm that multi-label ordering puts the dominant manipulation first.
#     # We define "dominant" mathematically by calculating the total modified pixel area per category.
#     cat_areas = {}
#     for r in regions:
#         if r.category_id in validated_cats:
#             cat_areas[r.category_id] = cat_areas.get(r.category_id, 0) + (r.w * r.h)
            
#     # Sort the localized categories by total bounding box area (largest first)
#     localized_sorted = sorted(
#         [c for c in validated_cats if c not in CATEGORY_ONLY_CLASSES],
#         key=lambda c: cat_areas.get(c, 0),
#         reverse=True
#     )
    
#     # Final ordering construction
#     final_ordered_categories = []
    
#     # If the whole page is fully AI-generated (C8), that is the ultimate dominant manipulation.
#     if "C8" in validated_cats:
#         final_ordered_categories.append("C8")
        
#     # Append the sorted localized categories
#     final_ordered_categories.extend(localized_sorted)
    
#     # If it's perfectly clean, C10 is the only category
#     if "C10" in validated_cats and "C10" not in final_ordered_categories:
#         final_ordered_categories.append("C10")

#     return {
#         "valid": True,
#         "final_categories": final_ordered_categories,
#         "dropped_categories": list(all_cats - set(final_ordered_categories)),
#         "notes": "Validation complete. Dominant category sorted mathematically by area."
#     }


# def build_page_analysis_result(
#     page: DocumentPage,
#     predicted_categories: List[str],
#     regions: List[DetectedRegion],
#     quality_summary: Optional[Dict[str, Any]] = None,
#     validation: Optional[Dict[str, Any]] = None,
# ) -> PageAnalysisResult:
#     """
#     Convert page-level outputs into a normalized analysis result object.

#     Intended responsibilities:
#     - Create a structured result for a single page.
#     - Preserve source link, page file name, original file name, page number,
#       predicted categories, localized regions, and debug notes.
#     - Keep the structure easy to serialize into submission JSON and YAML files.
#     """
#     # 1. Extract the final validated categories. 
#     # If validation failed or wasn't passed, default to C10 (No discrepancy found)
#     final_cats = ["C10"]
#     if validation and "final_categories" in validation:
#         final_cats = validation["final_categories"]
#     elif predicted_categories:
#         final_cats = predicted_categories
        
#     # 2. Compile debug information for trace logging
#     compiled_debug = {}
#     if quality_summary:
#         compiled_debug["quality"] = quality_summary
#     if validation:
#         compiled_debug["validation_details"] = validation
        
#     # 3. Construct the final normalized result object
#     return PageAnalysisResult(
#         source_link=page.source_link,
#         file_name=page.page_file_name,
#         original_file_name=page.original_file_name,
#         page_number=page.page_number,
#         predicted_categories=final_cats,
#         detected_regions=regions,
#         notes={"pipeline": "deterministic_cv_engine"},
#         debug=compiled_debug
#     )

## Output Builders

This section should follow the guideline requirements:
- JSON rows contain `link`, `file_name`, and `Category_ID`.
- YAML files are emitted per page using the page-style file name.
- Categories are joined with `||` in the JSON output when multiple apply.
- C8 and C10 do not require YAML. 

In [50]:
# =========================
# 7. OUTPUT / EXPORT HELPERS 
# =========================

def normalize_category_list(predicted_categories: List[str]) -> List[str]:
    valid = [c for c in predicted_categories if c in CATEGORY_IDS]
    seen = set()
    ordered = []
    for c in valid:
        if c not in seen:
            ordered.append(c)
            seen.add(c)
    return ordered


def build_json_row(result: PageAnalysisResult) -> Dict[str, Any]:
    categories = normalize_category_list(result.predicted_categories)
    category_string = "||".join(categories) if categories else "C10"

    return {
        "link": result.source_link,
        "file_name": result.file_name,
        "Category_ID": category_string,
    }


def detected_region_to_yaml_item(region: DetectedRegion) -> Dict[str, Any]:
    item: Dict[str, Any] = {
        "h": int(region.h),
        "w": int(region.w),
        "x": int(region.x),
        "y": int(region.y),
        "category_id": region.category_id,
    }

    if region.type is not None:
        item["type"] = region.type
    if region.stretch_factor is not None:
        item["stretch_factor"] = region.stretch_factor
    if region.header_source is not None:
        item["header_source"] = region.header_source
    if region.body_source is not None:
        item["body_source"] = region.body_source

    return item


def yaml_required_for_result(result: PageAnalysisResult) -> bool:
    categories = set(normalize_category_list(result.predicted_categories))
    if not categories:
        return False
    return not categories.issubset(CATEGORY_ONLY_CLASSES)


def yaml_name_for_page_file(page_file_name: str) -> str:
    return f"{Path(page_file_name).stem}.yaml"


def write_yaml_for_result(result: PageAnalysisResult, annotations_dir: Path) -> Optional[Path]:
    if not yaml_required_for_result(result):
        return None

    if yaml is None:
        raise ImportError("PyYAML is required to write annotation YAML files.")

    yaml_path = annotations_dir / yaml_name_for_page_file(result.file_name)
    yaml_items = [detected_region_to_yaml_item(region) for region in result.detected_regions]

    with open(yaml_path, "w", encoding="utf-8") as f:
        yaml.safe_dump(yaml_items, f, sort_keys=False, allow_unicode=True)

    return yaml_path


def write_submission_json(results: List[PageAnalysisResult], output_dir: Path) -> Path:
    json_rows = [build_json_row(r) for r in results]
    json_path = output_dir / "submission.json"

    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(json_rows, f, indent=2, ensure_ascii=False)

    return json_path


def write_optional_excel_summary(results: List[PageAnalysisResult], output_dir: Path) -> Optional[Path]:
    if pd is None:
        return None

    rows = []
    for r in results:
        rows.append({
            "link": r.source_link,
            "file_name": r.file_name,
            "Category_ID": "||".join(normalize_category_list(r.predicted_categories)) if r.predicted_categories else "C10",
            "region_count": len(r.detected_regions),
            "page_number": r.page_number,
            "original_file_name": r.original_file_name,
        })

    df = pd.DataFrame(rows)
    excel_path = output_dir / "submission_preview.xlsx"
    df.to_excel(excel_path, index=False)
    return excel_path


def export_all_outputs(results: List[PageAnalysisResult], output_dir: Path, annotations_dir: Path) -> Dict[str, Any]:
    annotation_paths = []
    for result in results:
        yaml_path = write_yaml_for_result(result, annotations_dir)
        if yaml_path is not None:
            annotation_paths.append(str(yaml_path))

    json_path = write_submission_json(results, output_dir)
    excel_path = write_optional_excel_summary(results, output_dir)

    return {
        "json_path": str(json_path),
        "yaml_paths": annotation_paths,
        "excel_preview_path": str(excel_path) if excel_path else None,
        "total_results": len(results),
    }

## Main Runner

The runner is fully implemented. It:
- onboards files,
- expands PDFs to page images,
- runs page-wise pipeline stages,
- creates final page results,
- exports JSON and YAML outputs.

The actual detection stages remain empty and will fall back to scaffold behavior until implemented.

In [ ]:
# =========================
# 9. MAIN RUNNER (FILLED)
# =========================

async def run_ps3_pipeline(
    input_dir: Path = INPUT_DIR,
    output_dir: Path = OUTPUT_DIR,
    annotations_dir: Path = ANNOTATIONS_DIR,
    debug_dir: Path = DEBUG_DIR,
) -> Dict[str, Any]:
    render_dir = debug_dir / "rendered_pages"
    pages = build_document_pages(input_dir, render_dir)

    results: List[PageAnalysisResult] = []

    for page in pages:
        # Quality Assessment
        print(f"Processing page {idx + 1} of {len(pages)}: {page.page_file_name}...")
        try:
            quality_summary = assess_page_quality(page)
            if quality_summary is None:
                quality_summary = {"status": "error"}
        except Exception as e:
            print(f"Error in assess_page_quality for {page.page_file_name}: {e}")
            quality_summary = {"status": "error"}

        # Categorization (Tier 4)
        try:
            predicted_categories = await detect_tampering_categories(page)
            if predicted_categories is None:
                predicted_categories = ["C10"]
        except Exception as e:
            print(f"Error in detect_tampering_categories for {page.page_file_name}: {e}")
            predicted_categories = ["C10"]

        predicted_categories = normalize_category_list(predicted_categories) or ["C10"]

        # Localization (Tiers 1, 2, 3)
        try:
            regions = await localize_tampered_regions(page, predicted_categories)
            if regions is None:
                regions = []
        except Exception as e:
            print(f"Error in localize_tampered_regions for {page.page_file_name}: {e}")
            regions = []

        # Postprocessing
        try:
            regions = postprocess_regions(regions, predicted_categories, page)
            if regions is None:
                regions = []
        except Exception as e:
            print(f"Error in postprocess_regions for {page.page_file_name}: {e}")
            regions = []

        # Enrichment
        try:
            regions = enrich_region_metadata(regions, predicted_categories, page)
            if regions is None:
                regions = []
        except Exception as e:
            print(f"Error in enrich_region_metadata for {page.page_file_name}: {e}")
            regions = []

        # Validation
        try:
            validation = validate_prediction(page, predicted_categories, regions)
            if validation is None:
                validation = {"final_categories": predicted_categories or ["C10"]}
        except Exception as e:
            print(f"Error in validate_prediction for {page.page_file_name}: {e}")
            validation = {"final_categories": predicted_categories or ["C10"]}

        # Build final result
        try:
            result = build_page_analysis_result(
                page=page,
                predicted_categories=predicted_categories,
                regions=regions,
                quality_summary=quality_summary,
                validation=validation,
            )
        except Exception as e:
            print(f"Error in build_page_analysis_result for {page.page_file_name}: {e}")
            result = PageAnalysisResult(
                source_link=page.source_link,
                file_name=page.page_file_name,
                original_file_name=page.original_file_name,
                page_number=page.page_number,
                predicted_categories=["C10"],
            )

        results.append(result)

    export_info = export_all_outputs(results, output_dir, annotations_dir)

    return {
        "pages": pages,
        "results": results,
        "export_info": export_info,
    }


## Display Helpers

In [52]:
# =========================
# 10. DISPLAY FINAL OUTPUTS (FILLED)
# =========================

def display_final_outputs(run_output: Dict[str, Any]) -> None:
    results: List[PageAnalysisResult] = run_output.get("results", [])
    export_info: Dict[str, Any] = run_output.get("export_info", {})

    print("=" * 80)
    print("PS3 PIPELINE RUN SUMMARY")
    print("=" * 80)
    print(f"Total page-level entries processed: {len(results)}")
    print(f"Submission JSON: {export_info.get('json_path')}")
    print(f"YAML files written: {len(export_info.get('yaml_paths', []))}")
    if export_info.get("excel_preview_path"):
        print(f"Excel preview: {export_info.get('excel_preview_path')}")
    print()

    preview_rows = [build_json_row(r) for r in results[:10]]
    if pd is not None and preview_rows:
        display(pd.DataFrame(preview_rows))
    else:
        print("Preview JSON rows:")
        print(json.dumps(preview_rows, indent=2))

    if export_info.get("yaml_paths"):
        print()
        print("Sample YAML files:")
        for yp in export_info["yaml_paths"][:10]:
            print("-", yp)

## Execution Cell



In [53]:
# !pip install openpyxl


In [ ]:
# =========================
# 11. EXECUTE
# =========================

run_output = await run_ps3_pipeline()
display_final_outputs(run_output)